# SpikeBit — Stage A ladder, remaining rung-3 runs on Colab (A100)

DECISIONS #15/#16/#17. Five runs, same commands as the local queue
(`ladder_run.sh`), outputs on Drive under `MyDrive/SpikeBit/{runs,checkpoints}/ladder/`:

| run | quantizer | bits | seed |
|---|---|---|---|
| `rung3m_a4h8_s1` | MinMax, clipped STE | 4, relu² output 8 | 1 |
| `rung3l_a4_s0` | LSQ, clipped STE | 4 | 0 |
| `rung3l_a4h8_s0` | LSQ, clipped STE | 4, relu² output 8 | 0 |
| `rung3l_a4_s1` | LSQ, clipped STE | 4 | 1 |
| `rung3l_a4h8_s1` | LSQ, clipped STE | 4, relu² output 8 | 1 |

Needs on Drive, under `MyDrive/SpikeBit/`: `bundle/spikebit_<commit>.tar.gz` (the code,
`git archive`), `checkpoints/ladder/rung1c_lr1e-3_s0/final.pt` (KL teacher), and optionally
`data/tinystories_packed_RWKV_rwkv-4-169m-pile_train_None.npz` (otherwise tokenized here,
~20 min). A finished run (`=== exit 0` in its `pipeline.log`) is skipped on re-run.

Runtime → Change runtime type → A100 GPU.

In [ ]:
import subprocess, torch, sys
name = torch.cuda.get_device_name(0)
print(name, torch.__version__, torch.cuda.get_device_capability(), sys.version.split()[0])
assert "A100" in name, f"expected an A100 runtime, got {name}"

In [ ]:
import glob, os
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/SpikeBit"
bundles = sorted(glob.glob(f"{DRIVE}/bundle/spikebit_*.tar.gz"), key=os.path.getmtime)
assert bundles, f"no bundle in {DRIVE}/bundle"
BUNDLE = bundles[-1]
print("bundle:", BUNDLE)
!rm -rf /content/SpikeBit && mkdir -p /content/SpikeBit && tar -xzf "$BUNDLE" -C /content/SpikeBit
%cd /content/SpikeBit
for sub in ("checkpoints", "runs", "data"):
    os.makedirs(f"{DRIVE}/{sub}", exist_ok=True)
    !rm -rf $sub && ln -s $DRIVE/$sub $sub
COMMIT = open("BUNDLE_COMMIT").read().strip()
print("commit:", COMMIT)

In [ ]:
# Colab ships its own torch; install the project on top of it.
!pip install -q -e . 2>&1 | tail -2
import hashlib
KL = "checkpoints/ladder/rung1c_lr1e-3_s0/final.pt"
h = hashlib.sha256()
with open(KL, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
assert h.hexdigest() == "04ea63897adf6dc7538db63848acd9c6a837883f85047acc3b8e94e9d4354c2c", "KL teacher checkpoint differs from the local one"
print("KL teacher OK")

In [ ]:
# Packed TinyStories (cached on Drive after the first time).
!python -c "from spiking_ternary_lm.training.data import tokenize_tinystories as t; c = t(tokenizer_id='RWKV/rwkv-4-169m-pile'); print(len(c.ids), 'tokens')"

In [ ]:
import datetime
KL_ARGS = ["--distill-checkpoint", KL]
TAIL = ["--seq-len", "128", "--batch-size", "16", "--reset-state-every", "1"]
def rung3(act_quant):
    return ["--weight-quant", "ternary", "--act-quant", act_quant, "--ffn-relu-sq", "--act-grid", "faithful",
            "--act-output", "dequant", "--act-ste", "clipped", "--lif-stateless", "--calibrate-batches", "8", *KL_ARGS]

def ladder_run(name, seed, lr, rung_args):
    run, ck = f"runs/ladder/{name}", f"checkpoints/ladder/{name}"
    os.makedirs(run, exist_ok=True); os.makedirs(ck, exist_ok=True)
    log = f"{run}/pipeline.log"
    if os.path.exists(log) and "=== exit 0" in open(log).read():
        print(name, "already done"); return 0
    cmd = ["python", "scripts/train_stage_a.py", "--teacher-id", "RWKV/rwkv-4-169m-pile",
           "--contiguous-rows", "--protocol-eval",
           "--seq-len", "64", "--batch-size", "32", "--grad-accum", "1", "--total-steps", "10000", "--warmup-steps", "300",
           "--lr-bitlinear", lr, "--lr-rwkv", lr, "--lr-embed-head", lr,
           "--log-every", "50", "--eval-every", "2000", "--ckpt-every", "100000", "--seed", str(seed),
           "--out-dir", ck, "--metrics-path", f"{run}/metrics.jsonl", *rung_args, *TAIL]
    with open(log, "a") as f:
        f.write(f"=== commit {COMMIT} host colab {name_gpu} start {datetime.datetime.now().astimezone().isoformat(timespec='seconds')}\n{' '.join(cmd)}\n")
    with open(f"{run}/train.log", "w") as out:
        rc = subprocess.run(cmd, stdout=out, stderr=subprocess.STDOUT).returncode
    with open(log, "a") as f:
        f.write(f"=== exit {rc} {datetime.datetime.now().astimezone().isoformat(timespec='seconds')}\n")
    print(name, "exit", rc, "|", subprocess.run(["tail", "-2", f"{run}/train.log"], capture_output=True, text=True).stdout)
    return rc

name_gpu = name.replace(" ", "_")
RUNS = [
    ("rung3m_a4h8_s1", 1, rung3("minmax") + ["--bits", "4", "--bits-ffn-hidden", "8"]),
    ("rung3l_a4_s0", 0, rung3("lsq") + ["--bits", "4"]),
    ("rung3l_a4h8_s0", 0, rung3("lsq") + ["--bits", "4", "--bits-ffn-hidden", "8"]),
    ("rung3l_a4_s1", 1, rung3("lsq") + ["--bits", "4"]),
    ("rung3l_a4h8_s1", 1, rung3("lsq") + ["--bits", "4", "--bits-ffn-hidden", "8"]),
]
for run_name, seed, args in RUNS:
    if ladder_run(run_name, seed, "3e-3", args) != 0:
        raise SystemExit(f"{run_name} failed: see runs/ladder/{run_name}/train.log")

In [ ]:
# Summary (same criterion as scripts/ladder_summary.py).
!python scripts/ladder_summary.py --runs runs/ladder --prefix rung3